## Data Collection: Google Trends via SerpApi

To measure how public interest in Netflix shifted around the password-sharing crackdown, this notebook pulls weekly search interest data for two terms — **"Netflix"** and **"cancel Netflix"** — across a subset of 14 countries spanning all five rollout waves.

**Why SerpApi instead of pytrends:** `pytrends` (the common unofficial Google Trends library) is currently unreliable due to aggressive rate-limiting from Google. SerpApi provides a stable, paid-tier-optional wrapper around Trends data instead.

**Why two separate calls per country, not one combined call:** Google Trends normalizes results on a 0–100 scale *relative to the highest-volume term in a single request*. Since "Netflix" is a far higher-volume search term than "cancel Netflix," querying them together compresses "cancel Netflix" down near zero — it looked flat not because interest was low, but because of how the terms were being scaled against each other. Querying each term separately fixes this by normalizing each one against its own peak.

**Why 14 countries instead of all ~40 available:** This subset preserves at least 2 countries from each of the 5 rollout waves — the minimum needed to keep the staggered-adoption structure intact — while staying well within the free API tier's monthly call limit and favoring larger markets with more reliable search volume.

**Output:** `netflix_trends_raw.csv` — one row per (country, week, search term), in long format. This gets merged with the treatment date table in the next step to build the final DiD dataset.

**Key limitation:** Because each term/country pair is normalized independently, raw interest values can't be compared directly *across* countries or *across* search terms — only a country's own trend over time, relative to its own pre-treatment baseline, is meaningful. This is a natural fit for difference-in-differences, which is designed around exactly that kind of within-unit comparison.

In [23]:
import pandas as pd 
import numpy as np 
from pytrends.request import TrendReq
import time
import serpapi
from dotenv import load_dotenv
import os




### We will only use a subset of the countries affected by the roll out (the more significant ones) to declutter the analysis.

In [11]:
treatment_df = pd.read_csv('data/netflix_rollout_dates.csv')

selected_countries = [
    # Wave 1 (March 2022)
    "Chile", "Peru",
    # Wave 2 (July 2022)
    "Argentina", "Guatemala",
    # Wave 3 (Feb 2023)
    "Canada", "Spain",
    # Wave 4 (May 2023) — bigger markets, better search volume
    "United States", "United Kingdom", "Germany", "Brazil", "Australia", "Mexico",
    # Wave 5 (July 2023)
    "India", "United Arab Emirates",
]

treatment_df = treatment_df[treatment_df['country'].isin(selected_countries)].reset_index(drop=True)
print(f"Subset: {len(treatment_df)} countries")
treatment_df



Subset: 14 countries


,country,country_code,rollout_date,wave,region
0,Chile,CL,2022-03-01,1,Americas
1,Peru,PE,2022-03-01,1,Americas
2,Argentina,AR,2022-07-01,2,Americas
3,Guatemala,GT,2022-07-01,2,Americas
4,Canada,CA,2023-02-08,3,Americas
5,Spain,ES,2023-02-08,3,Europe
6,United States,US,2023-05-23,4,Americas
7,Mexico,MX,2023-05-23,4,Americas
8,Brazil,BR,2023-05-23,4,Americas
9,United Kingdom,GB,2023-05-23,4,Europe


In [ ]:

load_dotenv()
API_KEY = os.getenv("SERPAPI_KEY")

client = serpapi.Client(api_key=API_KEY)

output_file = "netflix_trends_raw.csv"

# Resume support: skip countries already pulled
if os.path.exists(output_file):
    existing = pd.read_csv(output_file)
    done_combos = set(zip(existing['country_code'], existing['search_term']))
else:
    existing = pd.DataFrame()
    done_combos = set()

results = [existing] if not existing.empty else []
call_count = 0
MAX_CALLS = 200  # hard safety stop, well under your 250/month limit

In [20]:
search_terms = ["Netflix", "cancel Netflix"]

for _, row in treatment_df.iterrows():
    country_code = row['country_code']
    country_name = row['country']

    for term in search_terms:
        key = (country_code, term)
        if key in done_combos:  # track (country, term) pairs now, not just country
            print(f"Skipping (already pulled): {country_name} / {term}")
            continue

        if call_count >= MAX_CALLS:
            print("Hit self-imposed call limit, stopping.")
            break

        try:
            response = client.search({
                "engine": "google_trends",
                "q": term,  # single term now, not comma-separated
                "geo": country_code,
                "date": "2021-09-01 2023-12-31",
                "tz": "0",
                "data_type": "TIMESERIES",
            })
            call_count += 1

            timeline = response.get("interest_over_time", {}).get("timeline_data", [])
            if not timeline:
                print(f"No data: {country_name} / {term}")
                continue

            rows = []
            for point in timeline:
                date = point.get("date")
                timestamp = point.get("timestamp")
                for val in point.get("values", []):
                    rows.append({
                        "date": date,
                        "timestamp": timestamp,
                        "country": country_name,
                        "country_code": country_code,
                        "search_term": term,
                        "interest": val.get("extracted_value"),
                    })

            df = pd.DataFrame(rows)
            results.append(df)
            pd.concat(results, ignore_index=True).to_csv(output_file, index=False)
            print(f"Pulled: {country_name} / {term} (call {call_count}/{MAX_CALLS})")

        except Exception as e:
            print(f"Failed: {country_name} / {term} — {e}")

        time.sleep(1)

Pulled: Chile / Netflix (call 1/200)
Pulled: Chile / cancel Netflix (call 2/200)
Pulled: Peru / Netflix (call 3/200)
Pulled: Peru / cancel Netflix (call 4/200)
Pulled: Argentina / Netflix (call 5/200)
Pulled: Argentina / cancel Netflix (call 6/200)
Pulled: Guatemala / Netflix (call 7/200)
No data: Guatemala / cancel Netflix
Pulled: Canada / Netflix (call 9/200)
Pulled: Canada / cancel Netflix (call 10/200)
Pulled: Spain / Netflix (call 11/200)
Pulled: Spain / cancel Netflix (call 12/200)
Pulled: United States / Netflix (call 13/200)
Pulled: United States / cancel Netflix (call 14/200)
Pulled: Mexico / Netflix (call 15/200)
No data: Mexico / cancel Netflix
Pulled: Brazil / Netflix (call 17/200)
Pulled: Brazil / cancel Netflix (call 18/200)
Pulled: United Kingdom / Netflix (call 19/200)
Pulled: United Kingdom / cancel Netflix (call 20/200)
Pulled: Germany / Netflix (call 21/200)
Pulled: Germany / cancel Netflix (call 22/200)
Pulled: Australia / Netflix (call 23/200)
Pulled: Australia / c

In [21]:
trends_df = pd.read_csv(output_file)
print(f"Countries pulled: {trends_df['country'].nunique()}")
print(f"Search terms: {trends_df['search_term'].unique()}")
print(f"Date range: {trends_df['date'].min()} to {trends_df['date'].max()}")
trends_df.head(10)

Countries pulled: 14
Search terms: ['Netflix' 'cancel Netflix']
Date range: Apr 10 – 16, 2022 to Sep 5 – 11, 2021


,date,timestamp,country,country_code,search_term,interest
0,"Aug 29 – Sep 4, 2021",1630195200,Chile,CL,Netflix,73
1,"Sep 5 – 11, 2021",1630800000,Chile,CL,Netflix,71
2,"Sep 12 – 18, 2021",1631404800,Chile,CL,Netflix,84
3,"Sep 19 – 25, 2021",1632009600,Chile,CL,Netflix,88
4,"Sep 26 – Oct 2, 2021",1632614400,Chile,CL,Netflix,84
5,"Oct 3 – 9, 2021",1633219200,Chile,CL,Netflix,71
6,"Oct 10 – 16, 2021",1633824000,Chile,CL,Netflix,76
7,"Oct 17 – 23, 2021",1634428800,Chile,CL,Netflix,71
8,"Oct 24 – 30, 2021",1635033600,Chile,CL,Netflix,69
9,"Oct 31 – Nov 6, 2021",1635638400,Chile,CL,Netflix,78


In [22]:
trends_df[trends_df['search_term'] == "cancel Netflix"]['interest'].value_counts()

interest
0     845
41     23
57     22
39     21
35     20
     ... 
98      1
96      1
90      1
73      1
89      1
Name: count, Length: 85, dtype: int64